In [1]:
from pathlib import Path

import util

import automech

tag = util.notebook_prefix()

In [2]:
DATA_PATH = Path("../data")
CALC_PATH = Path(f"../{tag}/data/CKIN")

In [3]:
# Read mechanisms
par_mech0 = automech.io.read(DATA_PATH / "full_raw.json")
cal_mech0 = automech.io.read(DATA_PATH / f"{tag}.json")

In [4]:
# Add calculated thermo to mechanism object
*_, therm_file = CALC_PATH.glob("all_therm.ckin*")
cal_mech = automech.io.chemkin.update.thermo(cal_mech0, therm_file)

In [5]:
# Add calculated rates to mechanism object (use units of parent)
rate_files = list(CALC_PATH.glob("*.ckin"))
cal_mech = automech.set_rate_units(cal_mech, automech.rate_units(par_mech0))
cal_mech = automech.io.chemkin.update.rates(cal_mech, rate_files)

In [6]:
# Merge updated rates and thermo into parent mechanism
mech0 = automech.expand_parent_stereo(par_mech0, cal_mech)
mech = automech.update(mech0, cal_mech)

  0%|          | 0/3108 [00:00<?, ?it/s]

In [7]:
# Write
full0 = f"full_{tag}_control"
full = f"full_{tag}_calc"
part = f"{tag}_calc"
automech.io.write(mech0, DATA_PATH / f"{full0}.json")
automech.io.write(mech, DATA_PATH / f"{full}.json")
automech.io.write(cal_mech, DATA_PATH / f"{part}.json")
automech.io.chemkin.write.mechanism(mech0, DATA_PATH / "chemkin" / f"{full0}.dat")
automech.io.chemkin.write.mechanism(mech, DATA_PATH  / "chemkin" / f"{full}.dat")
automech.io.chemkin.write.mechanism(cal_mech, DATA_PATH  / "chemkin" / f"{part}.dat")

'ELEMENTS\n\nC\nH\nO\n\nEND\n\n\nSPECIES\n\nOH(4)         ! SMILES: [OH]                   AMChI: AMChI=1/HO/h1H\nHO2(8)        ! SMILES: [O]O                   AMChI: AMChI=1/HO2/c1-2/h2H\nC5H8(522)     ! SMILES: C1=CCCC1               AMChI: AMChI=1/C5H8/c1-2-4-5-3-1/h1-2H,3-5H2\nC5H8O(825)rs  ! SMILES: C1C[C@H]2[C@@H](C1)O2  AMChI: AMChI=1/C5H8O/c1-2-4-5(6-4)3-1/h4-5H,1-3H2/t4-,5+\nS(722)r0      ! SMILES: OO[C@H]1[CH]CCC1       AMChI: AMChI=1/C5H9O2/c6-7-5-1-2-3-4-5/h1,5-6H,2-4H2/t5-/m0/s1\nS(722)r1      ! SMILES: OO[C@@H]1[CH]CCC1      AMChI: AMChI=1/C5H9O2/c6-7-5-1-2-3-4-5/h1,5-6H,2-4H2/t5-/m1/s1\n\nEND\n\n\nTHERM\n\nOH(4)                   H   1O   1          G     200.0    3000.0  1000.0      1\n 3.33774379E+00-3.42517242E-05 5.48147391E-07-2.33181191E-10 2.96517411E-14    2\n 3.02742080E+03 3.14883237E+00 4.12321458E+00-3.27091148E-03 6.72273866E-06    3\n-6.14769971E-09 2.22076796E-12 2.84248919E+03-7.03916392E-01                   4\n\nHO2(8)                  H   1O   2      

In [8]:
# Visualize intersections with the calculated mechanism
int_par_mech0 = automech.intersection(
    par_mech0, cal_mech, stereo=False
)
int_cal_mech = automech.intersection(
    par_mech0, cal_mech, right=True, stereo=False
)
dif_cal_mech = automech.difference(
    par_mech0, cal_mech, right=True, stereo=False
)
int_mech = automech.intersection(mech, cal_mech, stereo=False)
print("\n1. Original (compare):")
print(automech.io.chemkin.write.reactions_block(int_par_mech0, frame=False))
print("\n2. Calculated (compare):")
print(automech.io.chemkin.write.reactions_block(int_cal_mech, frame=False))
if not automech.reaction_count(dif_cal_mech) == 0:
    print("\n3. Calculated (new):")
    print(automech.io.chemkin.write.reactions_block(dif_cal_mech, frame=False))
    print("\n\n4. Calculated (all together):")
    print(automech.io.chemkin.write.reactions_block(int_mech, frame=False))
    automech.display(int_par_mech0)
    automech.display(int_mech)


1. Original (compare):
S(722) = C5H8O(825) + OH(4)            1.000E+00      0.000      0.000
    PLOG  /                 0.0009870  9.620E+01      2.050      3.050/
    PLOG  /                 0.0009870  4.840E+35     -7.840      19.69/
    PLOG  /                  0.009870  1.290E+13     -1.370      5.946/
    PLOG  /                  0.009870  2.230E+38     -8.490      20.33/
    PLOG  /                   0.09870  3.220E+22     -4.170      8.727/
    PLOG  /                   0.09870  2.120E+32     -6.320      18.79/
    PLOG  /                    0.9870  1.280E+25     -4.670      10.25/
    PLOG  /                    0.9870  4.000E+24     -3.630      17.39/
    PLOG  /                     9.870  9.850E+17     -2.010      9.428/
    PLOG  /                     9.870  1.220E+79     -19.63      47.13/
    PLOG  /                     98.70  1.540E+24     -3.840      12.30/
    PLOG  /                     98.70  1.490E+00      3.720      4.676/
S(722) = C5H8(522) + HO2(8)            1.